# Phase III Kappa Parameter Study

This notebook studies the effect of the time derivative preconditioning parameter `κ` (`rkappa` in `main_solver.py`) for the lid-driven cavity solver. The project recommendation is to use at least a `65 x 65` mesh, so the default study below uses `65 x 65` nodes.


In [1]:
from pathlib import Path
import contextlib
import io
import os
import shutil
import time

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if Path.cwd().name != "start-code" and (Path.cwd() / "start-code").exists():
    os.chdir(Path.cwd() / "start-code")

plot_dir = Path("Phase III Kappa Study")
plot_dir.mkdir(exist_ok=True)

print(f"Working directory: {Path.cwd()}")
print(f"Plot output directory: {plot_dir.resolve()}")

Working directory: /Users/windy/mae4100-courseproject2/start-code
Plot output directory: /Users/windy/mae4100-courseproject2/start-code/Phase III Kappa Study


In [2]:
# Main study settings. Keep the mesh at 65x65 or higher for report-quality results.
NODE_COUNT = 65
REYNOLDS_NUMBER = 10.0
CFL = 0.5
TOLERANCE = 1e-10

# Start with a moderate range around the default rkappa = 0.5.
# If any case becomes unstable or very slow, rerun with fewer values near the best-performing region.
KAPPA_VALUES = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]

pd.DataFrame({
    "nodes": [f"{NODE_COUNT}x{NODE_COUNT}"] * len(KAPPA_VALUES),
    "Re": [REYNOLDS_NUMBER] * len(KAPPA_VALUES),
    "CFL": [CFL] * len(KAPPA_VALUES),
    "kappa": KAPPA_VALUES,
})

,nodes,Re,CFL,kappa
0,65x65,10.0,0.5,0.05
1,65x65,10.0,0.5,0.10
2,65x65,10.0,0.5,0.25
3,65x65,10.0,0.5,0.50
4,65x65,10.0,0.5,1.00
5,65x65,10.0,0.5,2.00


In [3]:
MAIN_PLOT_FILES = [
    "ucontour.png",
    "vcontour.png",
    "pcontour.png",
    "residualcomponent.png",
    "residual.png",
]


def replace_once(source, old, new, solver_path):
    if old not in source:
        raise RuntimeError(f"Could not find expected setting in {solver_path}: {old}")
    return source.replace(old, new, 1)


def format_solver_float(value):
    return f"{value:<16g}"


def patched_solver_source(kappa):
    solver_path = Path("main_solver.py")
    source = solver_path.read_text()

    replacements = {
        "imax = 9               # Number of points in the x-direction (use odd numbers only)":
            f"imax = {NODE_COUNT:<15}# Number of points in the x-direction (use odd numbers only)",
        "jmax = 9               # Number of points in the y-direction (use odd numbers only)":
            f"jmax = {NODE_COUNT:<15}# Number of points in the y-direction (use odd numbers only)",
        "cfl = 0.5              # CFL number used to determine time step":
            f"cfl = {format_solver_float(CFL)}# CFL number used to determine time step",
        "toler = 1e-10          # Tolerance for iterative residual convergence":
            f"toler = {format_solver_float(TOLERANCE)}# Tolerance for iterative residual convergence",
        "rkappa = 0.5           # Time derivative preconditioning constant":
            f"rkappa = {format_solver_float(kappa)}# Time derivative preconditioning constant",
        "Re = 10.0              # Reynolds number = rho*Uinf*L/rmu":
            f"Re = {format_solver_float(REYNOLDS_NUMBER)}# Reynolds number = rho*Uinf*L/rmu",
        "vectorize = False":
            "vectorize = True",
    }

    for old, new in replacements.items():
        source = replace_once(source, old, new, solver_path)

    return solver_path, source


def archive_solver_plots(case_label):
    for filename in MAIN_PLOT_FILES:
        src = Path(filename)
        if src.exists():
            dst = plot_dir / f"{case_label}_{filename}"
            if dst.exists():
                dst.unlink()
            shutil.move(str(src), str(dst))


def run_kappa_case(kappa):
    case_label = f"kappa_{kappa:g}".replace(".", "p")
    solver_path, source = patched_solver_source(kappa)
    namespace = {
        "__file__": str(solver_path.resolve()),
        "__name__": "__main__",
    }

    print(f"Running {NODE_COUNT}x{NODE_COUNT}, Re={REYNOLDS_NUMBER:g}, kappa={kappa:g}")
    start = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()) as captured_output:
        exec(compile(source, str(solver_path), "exec"), namespace)
    elapsed = time.perf_counter() - start
    archive_solver_plots(case_label)
    plt.close("all")

    u = namespace["u"].copy()
    imax, jmax, _ = u.shape
    i_mid = (imax - 1) // 2
    j_mid = (jmax - 1) // 2
    x_coords = np.linspace(namespace["xmin"], namespace["xmax"], imax)
    y_coords = np.linspace(namespace["ymin"], namespace["ymax"], jmax)

    return {
        "kappa": kappa,
        "label": case_label,
        "nodes": NODE_COUNT,
        "Re": REYNOLDS_NUMBER,
        "CFL": CFL,
        "elapsed_time_sec": elapsed,
        "iterations": namespace["n"],
        "converged": namespace["isConverged"],
        "final_conv": namespace["conv"],
        "final_residual": np.array(namespace["res"], copy=True),
        "conv_history": np.array(namespace["convVector"], copy=True),
        "residual_history": np.array(namespace["resPMatrix"], copy=True),
        "x": x_coords,
        "y": y_coords,
        "u_vertical_centerline": u[i_mid, :, 1].copy(),
        "v_horizontal_centerline": u[:, j_mid, 2].copy(),
        "p_vertical_centerline": u[i_mid, :, 0].copy(),
        "p_horizontal_centerline": u[:, j_mid, 0].copy(),
        "captured_output": captured_output.getvalue(),
    }

In [4]:
results = []
for kappa in KAPPA_VALUES:
    results.append(run_kappa_case(kappa))

summary_table = pd.DataFrame([
    {
        "kappa": result["kappa"],
        "nodes": f"{result['nodes']}x{result['nodes']}",
        "Re": result["Re"],
        "CFL": result["CFL"],
        "iterations": result["iterations"],
        "converged": result["converged"],
        "final_conv": result["final_conv"],
        "wall_time_s": result["elapsed_time_sec"],
        "continuity_residual": result["final_residual"][0],
        "x_momentum_residual": result["final_residual"][1],
        "y_momentum_residual": result["final_residual"][2],
    }
    for result in results
]).sort_values("kappa")

summary_table

Running 65x65, Re=10, kappa=0.05


KeyboardInterrupt: 

In [ ]:
fontsize = 12

plt.figure(figsize=(7, 5))
for result in results:
    history = np.maximum(result["conv_history"], 1e-300)
    plt.semilogy(history, linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("iteration", fontsize=fontsize)
plt.ylabel("overall residual", fontsize=fontsize)
plt.title(f"Residual convergence on {NODE_COUNT}x{NODE_COUNT} mesh", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_residual_histories.png", dpi=300)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(summary_table["kappa"], summary_table["iterations"], "o-", linewidth=2)
axes[0].set_xscale("log")
axes[0].set_xlabel("κ", fontsize=fontsize)
axes[0].set_ylabel("iterations", fontsize=fontsize)
axes[0].set_title("Iterations to convergence", fontsize=fontsize)

axes[1].plot(summary_table["kappa"], summary_table["wall_time_s"], "o-", linewidth=2)
axes[1].set_xscale("log")
axes[1].set_xlabel("κ", fontsize=fontsize)
axes[1].set_ylabel("wall time (s)", fontsize=fontsize)
axes[1].set_title("Wall time to convergence", fontsize=fontsize)

for ax in axes:
    ax.tick_params(labelsize=fontsize)
    ax.grid(True, which="both", alpha=0.25)

plt.tight_layout()
plt.savefig(plot_dir / "kappa_efficiency_summary.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for result in results:
    plt.plot(result["u_vertical_centerline"], result["y"], linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("u velocity (m/s)", fontsize=fontsize)
plt.ylabel("y (m)", fontsize=fontsize)
plt.title("Vertical centerline u velocity", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_vertical_centerline_u.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
for result in results:
    plt.plot(result["x"], result["v_horizontal_centerline"], linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("x (m)", fontsize=fontsize)
plt.ylabel("v velocity (m/s)", fontsize=fontsize)
plt.title("Horizontal centerline v velocity", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_horizontal_centerline_v.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
for result in results:
    plt.plot(result["p_vertical_centerline"], result["y"], linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("p (N/m^2)", fontsize=fontsize)
plt.ylabel("y (m)", fontsize=fontsize)
plt.title("Vertical centerline pressure", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_vertical_centerline_pressure.png", dpi=300)
plt.show()

In [ ]:
# Quantify whether kappa changes the converged solution.
# The baseline is the course default kappa = 0.5 when included in KAPPA_VALUES.
baseline = next((result for result in results if np.isclose(result["kappa"], 0.5)), results[0])

profile_difference_table = pd.DataFrame([
    {
        "kappa": result["kappa"],
        "baseline_kappa": baseline["kappa"],
        "max_abs_delta_u_vertical_centerline": np.max(np.abs(result["u_vertical_centerline"] - baseline["u_vertical_centerline"])),
        "max_abs_delta_v_horizontal_centerline": np.max(np.abs(result["v_horizontal_centerline"] - baseline["v_horizontal_centerline"])),
        "max_abs_delta_p_vertical_centerline": np.max(np.abs(result["p_vertical_centerline"] - baseline["p_vertical_centerline"])),
    }
    for result in results
]).sort_values("kappa")

profile_difference_table